# Подготовка данных

## Импортирование библиотек

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

## Загрузка датасета

In [ ]:
RAW_PATH = Path("../data/raw/ipc_mes_09-2026.xlsx")
sheet_name = "01"
df_raw = pd.read_excel(RAW_PATH, sheet_name=sheet_name, header=None)

## Отбор ячеек со значениями ИПЦ

In [ ]:
EXPECTED_YEAR_ROW = 3
EXPECTED_MONTH_START = 5

year_row = df_raw.iloc[EXPECTED_YEAR_ROW, :]
year_cols = [
    i for i, v in enumerate(year_row)
    if pd.notna(v) and isinstance(v, (int, float)) and 1990 <= v <= 2030
]
years = [int(year_row[i]) for i in year_cols]

print(f"Годы: {len(years)}, от {years[0]} до {years[-1]}")


df_block = df_raw.iloc[EXPECTED_MONTH_START:EXPECTED_MONTH_START + 12, year_cols].copy()
df_block.columns = [str(y) for y in years]

months = (
    df_raw.iloc[EXPECTED_MONTH_START:EXPECTED_MONTH_START + 12, 0]
    .astype(str).str.strip().str.lower().values
)
df_block.insert(0, "Month", months)

## Формирование датасета со значениями ИПЦ

In [ ]:
df_long = df_block.melt(id_vars="Month", var_name="Year", value_name="cpi_mom")

month_map = {
    "январь": "01", "февраль": "02", "март": "03", "апрель": "04",
    "май": "05", "июнь": "06", "июль": "07", "август": "08",
    "сентябрь": "09", "октябрь": "10", "ноябрь": "11", "декабрь": "12",
}
df_long["Month_num"] = df_long["Month"].map(month_map)

df_long["Date"] = pd.to_datetime(df_long["Year"] + "-" + df_long["Month_num"] + "-01")

df_long["cpi_mom"] = pd.to_numeric(df_long["cpi_mom"], errors="coerce")

df_final = (
    df_long[["Date", "cpi_mom"]]
    .sort_values("Date")
    .dropna()
    .reset_index(drop=True)
)
df_final = df_final[df_final["Date"] <= "2022-03-01"].reset_index(drop=True)

print(f"\nИтоговый размер: {df_final.shape}")
df_final.head()

## Сохранение датасета на диск

In [ ]:
PROCESSED_PATH = Path("../data/processed/cpi_monthly.csv")
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

df_final.to_csv(PROCESSED_PATH, index=False)

print(f"Сохранено: {PROCESSED_PATH.resolve()}")
print(f"Строк: {len(df_final)}")

## Рост цен с декабря 1990 года

In [ ]:
cumulative_cpi = (df_final["cpi_mom"] / 100).prod()
print(f"Кумулятивный рост цен: x{cumulative_cpi:,.0f} (дек 1990 - мар 2022)")